# analysis datasets

In [1]:
import sys
sys.path.append("/home/icclab/Documents/lqw/sam3_finetune_lora")  # Add project root to path
    

import argparse
import json
import os
from pathlib import Path

import numpy as np
import pycocotools.mask as mask_utils  # Required for RLE mask decoding in COCO dataset
from PIL import Image as PILImage
from sam3.train.data.collator import collate_fn_api

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from sam3.train.data.sam3_image_dataset import (
    Datapoint,
    FindQueryLoaded,
    Image,
    InferenceMetadata,
    Object,
)

/home/icclab/miniconda3/envs/Ref/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class COCOSegmentDataset(Dataset):
    """
    用于加载 COCO 格式分割数据的数据集类。

    该类负责读取 COCO JSON 标注文件，加载图像，并将标注（边界框和多边形/RLE 掩码）
    转换为 SAM3 模型训练所需的格式。

    属性:
        data_dir (Path): 包含训练/验证/测试文件夹的根目录。
        split (str): 数据拆分类型 ('train', 'valid', 'test')。
        resolution (int): 图像缩放的目标分辨率，默认为 1008。
    """

    def __init__(self, data_dir: str, split: str = "train"):
        """
        初始化数据集。

        参数:
            data_dir (str): 根目录路径。
            split (str): 'train', 'valid' 或 'test'。

        异常:
            FileNotFoundError: 如果在指定路径找不到 COCO 标注文件。
        """
        self.data_dir = Path(data_dir)
        self.split = split
        self.split_dir = self.data_dir / split

        # 加载 COCO 标注
        ann_file = self.split_dir / "_annotations.coco.json"
        if not ann_file.exists():
            raise FileNotFoundError(f"未找到 COCO 标注文件: {ann_file}")

        with open(ann_file) as f:
            self.coco_data = json.load(f)

        # 构建索引: image_id -> 图像信息
        self.images = {img["id"]: img for img in self.coco_data["images"]}
        self.image_ids = sorted(list(self.images.keys()))

        # 构建索引: image_id -> 标注列表
        self.img_to_anns = {}
        for ann in self.coco_data["annotations"]:
            img_id = ann["image_id"]
            if img_id not in self.img_to_anns:
                self.img_to_anns[img_id] = []
            self.img_to_anns[img_id].append(ann)

        # 加载类别信息
        self.categories = {
            cat["id"]: cat["name"] for cat in self.coco_data["categories"]
        }
        print(f"已加载 COCO 数据集: {split} 分组")
        print(f"  图像总数: {len(self.image_ids)}")
        print(f"  标注总数: {len(self.coco_data['annotations'])}")
        print(f"  类别列表: {self.categories}")

        self.resolution = 1008
        self.transform = v2.Compose(
            [
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            ]
        )

    def __len__(self) -> int:
        """返回数据集中的图像总数。"""
        return len(self.image_ids)

    def __getitem__(self, idx: int) -> Datapoint:
        """
        获取指定索引的数据点。

        读取图像并处理其关联的所有标注，转换为 SAM3 内部使用的 Datapoint 对象。

        参数:
            idx (int): 索引。

        返回:
            Datapoint: 包含图像数据、对象标注和查询信息的对象。
        """
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]

        # 加载图像
        img_path = self.split_dir / img_info["file_name"]
        pil_image = PILImage.open(img_path).convert("RGB")
        orig_w, orig_h = pil_image.size

        # 缩放图像
        pil_image = pil_image.resize(
            (self.resolution, self.resolution), PILImage.BILINEAR
        )

        # 转换为张量
        image_tensor = self.transform(pil_image)

        # 获取该图像的标注
        annotations = self.img_to_anns.get(img_id, [])

        objects = []
        object_class_names = []

        # 缩放因子
        scale_w = self.resolution / orig_w
        scale_h = self.resolution / orig_h

        for i, ann in enumerate(annotations):
            # 获取边界框 - COCO 格式为 [x, y, width, height]
            bbox_coco = ann.get("bbox", None)
            if bbox_coco is None:
                continue

            # 从 category_id 获取类别名称
            category_id = ann.get("category_id", 0)
            class_name = self.categories.get(category_id, "object")
            object_class_names.append(class_name)

            # 转换为归一化的 CxCyWH 格式
            x, y, w, h = bbox_coco
            # print(f"原始 COCO bbox: {bbox_coco} (x, y, w, h)", type(w), type(h))
            cx = x + float(w) / 2.0
            cy = y + float(h) / 2.0

            # 缩放并归一化到 [0, 1]
            box_tensor = torch.tensor(
                [
                    cx * scale_w / self.resolution,
                    cy * scale_h / self.resolution,
                    float(w) * scale_w / self.resolution,
                    float(h) * scale_h / self.resolution,
                ],
                dtype=torch.float32,
            )

            # 处理分割掩码 (多边形或 RLE 格式)
            segment = None
            segmentation = ann.get("segmentation", None)

            if segmentation:
                try:
                    if isinstance(segmentation, dict):
                        # RLE 格式
                        mask_np = mask_utils.decode(segmentation)
                    elif isinstance(segmentation, list):
                        # 多边形格式
                        rles = mask_utils.frPyObjects(segmentation, orig_h, orig_w)
                        rle = mask_utils.merge(rles)
                        mask_np = mask_utils.decode(rle)
                    else:
                        segment = None
                        continue

                    # 缩放到模型分辨率
                    mask_t = torch.from_numpy(mask_np).float().unsqueeze(0).unsqueeze(0)
                    mask_t = torch.nn.functional.interpolate(
                        mask_t, size=(self.resolution, self.resolution), mode="nearest"
                    )
                    segment = mask_t.squeeze() > 0.5  # [1008, 1008] 布尔张量

                except Exception as e:
                    print(f"警告: 处理图像 {img_id} 掩码时出错: {e}")
                    segment = None

            obj = Object(
                bbox=box_tensor,
                area=(box_tensor[2] * box_tensor[3]).item(),
                object_id=i,
                segment=segment,
            )
            objects.append(obj)

        image_obj = Image(
            data=image_tensor, objects=objects, size=(self.resolution, self.resolution)
        )

        from collections import defaultdict

        # 按类别名称分组对象 ID
        class_to_object_ids = defaultdict(list)
        for obj, class_name in zip(objects, object_class_names):
            class_to_object_ids[class_name.lower()].append(obj.object_id)

        # 每个类别创建一个查询
        queries = []
        if len(class_to_object_ids) > 0:
            for query_text, obj_ids in class_to_object_ids.items():
                query = FindQueryLoaded(
                    query_text=query_text,
                    image_id=0,
                    object_ids_output=obj_ids,
                    is_exhaustive=True,
                    query_processing_order=0,
                    inference_metadata=InferenceMetadata(
                        coco_image_id=img_id,
                        original_image_id=img_id,
                        original_category_id=0,
                        original_size=(orig_h, orig_w),
                        object_id=-1,
                        frame_index=-1,
                    ),
                )
                queries.append(query)
        else:
            # 无标注时，创建一个通用查询
            query = FindQueryLoaded(
                query_text="object",
                image_id=0,
                object_ids_output=[],
                is_exhaustive=True,
                query_processing_order=0,
                inference_metadata=InferenceMetadata(
                    coco_image_id=img_id,
                    original_image_id=img_id,
                    original_category_id=0,
                    original_size=(orig_h, orig_w),
                    object_id=-1,
                    frame_index=-1,
                ),
            )
            queries.append(query)

        return Datapoint(
            find_queries=queries, images=[image_obj], raw_images=[pil_image]
        )

In [3]:
def collate_fn(batch):
	return collate_fn_api(batch, dict_key="input", with_seg_masks=True)

In [ ]:
train_ds = COCOSegmentDataset(data_dir="/home/icclab/Documents/lqw/DatasetMMF/crackDetection", \
							  split="train")
train_loader = DataLoader(
	train_ds,
	batch_size=4,
	shuffle=True,  # Shuffle is handled by the sampler
	collate_fn=collate_fn,
	num_workers=2,
	pin_memory=True,
)

已加载 COCO 数据集: train 分组
  图像总数: 338
  标注总数: 404
  类别列表: {0: 'crack', 1: 'crack'}


In [34]:
for batch_dict in train_loader:
	input_batch = batch_dict["input"]
	image = input_batch.img_batch
	text = input_batch.find_text_batch
	inputs = input_batch.find_inputs
	targets = input_batch.find_targets
	metadatas = input_batch.find_metadatas
	raw_images = input_batch.raw_images

	print(image.shape)
	print(len(text), text)
	print(len(inputs), inputs)
	print(len(targets), targets[0])
	print(len(metadatas), metadatas)
	print(len(raw_images), raw_images[0].size)

	# print(input_batch)
	break  # 仅测试一个批次

torch.Size([2, 3, 1008, 1008])
1 ['crack']
1 [FindStage(img_ids=tensor([0, 1]), text_ids=tensor([0, 0]), input_boxes=tensor([], size=(0, 2, 4)), input_boxes_mask=tensor([], size=(2, 0), dtype=torch.bool), input_boxes_label=tensor([], size=(0, 2), dtype=torch.int64), input_points=tensor([], size=(2, 0, 257)), input_points_mask=tensor([], size=(2, 0), dtype=torch.bool), object_ids=[[0, 1], [0, 1]])]
1 BatchedFindTarget(num_boxes=tensor([2, 2]), boxes=tensor([[0.7599, 0.4714, 0.4802, 0.1498],
        [0.5925, 0.2048, 0.8150, 0.3921],
        [0.1872, 0.6167, 0.3744, 0.1057],
        [0.6388, 0.7709, 0.5022, 0.4581]]), boxes_padded=tensor([[[0.7599, 0.4714, 0.4802, 0.1498],
         [0.5925, 0.2048, 0.8150, 0.3921]],

        [[0.1872, 0.6167, 0.3744, 0.1057],
         [0.6388, 0.7709, 0.5022, 0.4581]]]), repeated_boxes=tensor([]), segments=tensor([[[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  .

In [7]:
input = np.array([1, 2, 3])
b = {"value": np.array([1, 2, 3])}

In [8]:
a = {"img_batch_all_stages": input}

In [9]:
a.update(b)

In [ ]:


def convert_predictions_to_coco_format(
    predictions_list,
    image_ids,
    resolution=288,
    score_threshold=0.0,
    merge_overlaps=True,
    iou_threshold=0.3,
    debug=False,
):
    """
    将模型预测结果转换为 COCO 格式。

    优化策略：掩码保持在模型原生输出分辨率 (288x288)，对应的 Ground Truth 也会被降采样，
    这样可以显著加快计算速度，无需进行大图上采样。

    参数:
        predictions_list (list): 模型输出的预测字典列表。
        image_ids (list): 与预测对应的图像 ID 列表。
        resolution (int, 可选): 掩码分辨率，默认 288。
        score_threshold (float, 可选): 置信度阈值。
        merge_overlaps (bool, 可选): 是否合并重叠预测，默认 True。
        iou_threshold (float, 可选): 合并 IoU 阈值，默认 0.3。
        debug (bool, 可选): 是否打印调试信息。

    返回:
        list: COCO 格式的预测字典列表。
    """
    coco_predictions = []
    pred_id = 0

    for img_id, preds in zip(image_ids, predictions_list):
        if preds is None or len(preds.get("pred_logits", [])) == 0:
            continue

        # 提取预测结果
        logits = preds["pred_logits"]  # [num_queries, 1]
        boxes = preds["pred_boxes"]  # [num_queries, 4]
        masks = preds["pred_masks"]  # [num_queries, H, W]

        scores = torch.sigmoid(logits).squeeze(-1)  # [num_queries]

        # 按置信度阈值过滤
        valid_mask = scores > score_threshold
        num_before = len(scores)
        scores = scores[valid_mask]
        boxes = boxes[valid_mask]
        masks = masks[valid_mask]

        if debug and img_id == image_ids[0]:  # 仅对首张图进行调试打印
            print(
                f"  图像 {img_id}: {num_before} 个查询 -> 过滤后剩余 {len(scores)} 个 (阈值={score_threshold})"
            )

        # 转换为二值掩码
        binary_masks = (torch.sigmoid(masks) > 0.5).cpu()

        # 合并重叠预测，避免过度分割惩罚
        if merge_overlaps and len(binary_masks) > 0:
            num_before_merge = len(binary_masks)
            binary_masks, scores, boxes = merge_overlapping_masks(
                binary_masks, scores.cpu(), boxes.cpu(), iou_threshold=iou_threshold
            )
            if debug and img_id == image_ids[0]:
                print(
                    f"  合并前 {num_before_merge} 个 -> 合并后 {len(binary_masks)} 个 (IoU 阈值={iou_threshold})"
                )

        # 编码为 RLE (原生分辨率下非常快)
        if len(binary_masks) > 0:
            mask_areas = binary_masks.flatten(1).sum(1)

            if debug and img_id == image_ids[0]:
                print(f"  掩码形状: {binary_masks.shape}")
                print(
                    f"  面积统计: 最小={mask_areas.min():.0f}, 最大={mask_areas.max():.0f}, 平均={mask_areas.float().mean():.0f}"
                )

            rles = rle_encode(binary_masks)

            for _idx, (rle, score, box) in enumerate(
                zip(rles, scores.cpu().tolist(), boxes.cpu().tolist())
            ):
                # 将归一化的 CxCyWH 转换为像素坐标下的 [x, y, w, h]
                cx, cy, w, h = box
                x = (cx - w / 2) * resolution
                y = (cy - h / 2) * resolution
                w = w * resolution
                h = h * resolution

                coco_predictions.append(
                    {
                        "image_id": int(img_id),
                        "category_id": 1,
                        "segmentation": rle,
                        "bbox": [float(x), float(y), float(w), float(h)],
                        "score": float(score),
                        "id": pred_id,
                    }
                )
                pred_id += 1

    return coco_predictions


{'img_batch_all_stages': array([1, 2, 3]), 'value': array([1, 2, 3])}